# GCOL-OBS — PATCH.10H.0R.2
## Full X22/X24 Metadata and Amendment Decision Gate — FAST

**Project:** ALMA `2015.1.00147.S` · `NGC1333_IRAS4A` · members `X22`, `X24`.

**Scope:** consolidate exact original pre-registration, archive inventories, README/QA2, the two small continuum FITS headers, the archived twelve primary cubes' FITS headers, direct X24 SPW25 header, and user-supplied X22 SPW25 Colab header + hash report. Produce an amendment *recommendation*, **not** an authorization for line-signal testing.

**Non-negotiable:** no analysis of line intensities, no alteration of velocity windows or thresholds, `GVH_USED=False`. We do not confuse catalogue presence, reported header, independent raw-byte verification, and physical discovery.

Four research levels: (1) CHH/GVH hypotheses (none tested here); (2) established FITS/ALMA metadata conventions; (3) executed metadata and arithmetic gates; (4) precise scientific verdict.

### Provenance and freeze

- Original `PATCH.10H.0` preregistration is **immutable**. Its expected SPW IDs are historically mismatched to the observed FITS `RESTFRQ`, but not silently overwritten.
- Nominal frozen velocity windows: **6.40–7.00 km/s** (core), **7.70–8.30 km/s** (foreground test). `spw25_primary_test=False` in the original document.
- Archive `X22` and `X24` currently contain the recovered products. Earlier `PATCH10H0_PASS=False` and `NEXT_PATCH_AUTHORIZED=False` are **historic** and remain untouched.
- X22 SPW25 **large raw file not ingested in this local audit** (1.349 GB; connector limit 256 MB). Its reported FITS header/SHA come from the user's Colab notebook with saved stream outputs (execution counters absent). This distinction is kept throughout.
- Lines' labels are those of the original contract; a FITS `RESTFRQ` agreement certifies **metadata alignment**, not chemical/species identification from emission/absorption.

In [1]:
import csv, hashlib, json, math, re
from pathlib import Path
import numpy as np

BASE = Path('/mnt/data') if Path('/mnt/data/PATCH10H0_preregistration_contract.json').exists() else Path.cwd()
OUTPUT = Path('/mnt/data') if Path('/mnt/data').exists() else Path.cwd()
STEM = 'GCOL_OBS_PATCH10H0R2_Full_X22_X24_Metadata_And_Amendment_Decision_Gate_FAST'
C_KM_S = 299792.458
GVH_USED = False
LINE_AMPLITUDES_INSPECTED = False
ALLOW_VELOCITY_WINDOW_RETUNING = False
AMENDMENT_ADOPTED = False
LINE_TEST_AUTHORIZED = False
assert not any([GVH_USED, LINE_AMPLITUDES_INSPECTED, ALLOW_VELOCITY_WINDOW_RETUNING, AMENDMENT_ADOPTED, LINE_TEST_AUTHORIZED])
checks = {}
def gate(key, condition, evidence=''):
    checks[key] = {'passed': bool(condition), 'evidence': str(evidence)}
    assert condition, f'FAIL {key}: {evidence}'
    print('PASS', key)

def sha256_file(path, chunk_size=2**20):
    h=hashlib.sha256()
    with open(path,'rb') as fh:
        for chunk in iter(lambda:fh.read(chunk_size),b''):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_SOURCE_SHA = {'contract': 'ec019b74dfec41fba05ccc78170a98324386e02cdf5f9f2267beb4f145a54831', 'h0_status': '07d3a5f04abf024fe94c8f0b7b81bae0b732fca7bc68c3194bd26d347b6bba96', 'old_headers': 'bc54b9eedad3ddc444fdea2e7ac7efea1b403c37b1c5ee6023de5cce9408fcd3', 'old_hashes': '27888131590d0a20134451f5d209ecaa08d976e990d87ce54bf6babac1c281bb', 'old_mapping': '7a6b2a805f33af806aac5f3b9511d7da666915ce0111775bc2ad335d67cbb0b4', 'r1_review': 'fe2db82ed93e90b1167d1cbecb54fe671350bdab43a8407e113b08d2f829cc97', 'readme': '3e6a7a99f3c1c26951b0cb1665df62f70f69633a9d33a4b8553f4538178c8b46', 'cont22': 'd1f0bd1d2c38e9af627c88515dc1dfdcf5ea4a2e5381add53594b8de1c1fc70e', 'cont24': 'e858a7fd8fafde20799082c60c37c1c17e0d0f3b7fbe402a8dcf75a781c63b1c', 'spw25x24': '4c1d7549d3cdc44527ef5c234ff45b4e0fa8205915bf2bf8691704252612f4da'}
INPUT_FILENAMES = {'contract': 'PATCH10H0_preregistration_contract.json', 'h0_status': 'PATCH10H0_status.json', 'old_headers': 'PATCH10H0_fits_header_audit.csv', 'old_hashes': 'PATCH10H0_required_files_sha256.csv', 'old_mapping': 'PATCH10H0R_frequency_mapping_evidence.csv', 'r1_review': 'GCOL_OBS_PATCH10H0R1_X22_SPW25_Independent_Review_2026-11-20.json', 'readme': 'member.uid___A001_X2f6_X22.README.txt', 'cont22': 'member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw25_27_29_31_33_35_37.cont.I.pbcor.fits.bin', 'cont24': 'member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw25_27_29_31_33_35_37.cont.I.pbcor.fits.bin', 'spw25x24': 'member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw25.cube.I.pbcor.fits.bin'}
SOURCE_SNAPSHOT = {'contract': {'patch': 'GCOL-OBS-0.3.2.1.1-PATCH.10H.0', 'project_code': '2015.1.00147.S', 'members': ['X22', 'X24'], 'source': 'NGC1333_IRAS4A', 'velocity_windows_kms': {'core': [6.4, 7.0], 'foreground_test': [7.7, 8.3]}, 'primary_spws': [27, 29, 31, 33, 35, 37], 'expected_spw_mapping': {'27': {'label': 'H2CO', 'rest_GHz_expected': [351.76864]}, '29': {'label': 'HNC', 'rest_GHz_expected': [362.6303]}, '31': {'label': 'H2CO', 'rest_GHz_expected': [362.73605]}, '33': {'label': 'H2CO', 'rest_GHz_expected': [363.94589]}, '35': {'label': 'H2CO', 'rest_GHz_expected': [364.10325]}, '37': {'label': 'H2CO', 'rest_GHz_expected': [364.27514, 364.28888]}}, 'spw25_primary_test': False, 'line_signal_inspection_allowed_in_10H0': False, 'threshold_retuning_after_unblinding_allowed': False, 'window_shift_after_unblinding_allowed': False, 'GVH_used': False}, 'h0_status': {'PATCH': 'GCOL-OBS-0.3.2.1.1-PATCH.10H.0', 'PROJECT_CODE': '2015.1.00147.S', 'ARCHIVE_INVENTORY_PASS': True, 'README_PRESENT': False, 'X22_REQUIRED_SET_PASS': False, 'X24_REQUIRED_SET_PASS': True, 'HEADER_READABILITY_PASS': True, 'PRIMARY_CUBE_SPECTRAL_AXIS_PASS': True, 'PRIMARY_CUBE_RESOLUTION_ESTIMATED_PASS': True, 'PREREGISTRATION_FROZEN': True, 'FROZEN_WINDOWS_KMS': {'core': [6.4, 7.0], 'foreground_test': [7.7, 8.3]}, 'PRIMARY_SPWS': [27, 29, 31, 33, 35, 37], 'LINE_SIGNAL_INSPECTED_IN_10H0': False, 'THRESHOLDS_CHANGED': False, 'WINDOWS_CHANGED': False, 'IRAS4_FOREGROUND_LAYER_CERTIFIED': False, 'FRONT_BACK_LAYER_ORDER_DERIVED': False, 'GCOL_OBS_0_3_2_1_1_CLOSED': False, 'GCOL_OBS_0_3_2_1_CLOSED': False, 'GVH_USED': False, 'NEXT_PATCH_AUTHORIZED': False, 'NEXT_PATCH': 'GCOL-OBS-0.3.2.1.1-PATCH.10H.1 — continuum geometry, source-registration & spectral-axis certification', 'PATCH10H0_PASS': False, 'PATCH10H0_DECISION': 'ABSTAIN_ARCHIVE_OR_PROVENANCE_NOT_READY'}, 'r1_review': {'review_id': 'GCOL-OBS-PATCH.10H.0R.1-REVIEW-2026-11-20', 'source': 'user-uploaded executed-output Jupyter notebook', 'source_file': 'GCOL_OBS_PATCH10H0R1_X22_SPW25_Header_SHA256_Blind_Verification(2).ipynb', 'source_notebook_sha256': '5dd66ce58513c1042c34c1edf0ce6c175ca239276d5f964789aaf8df2c2554ea', 'code_cells': 4, 'code_cells_with_saved_stream_output': 4, 'execution_counts_all_missing': True, 'independently_read_full_fits_cube': False, 'original_archival_sha256_verified': False, 'reported': {'file_size_bytes': 1349115840, 'sha256_from_colab_output': '67fd0dee3db8f9ba97db1f63d0eb655582065d1f6f57f16e1d046a3eae728a78', 'fits_header': {'header_only': True, 'naxis': 4, 'axes': [{'axis': 1, 'ctype': 'RA---SIN', 'cunit': 'deg', 'naxis': 840, 'crval': 52.29375, 'cdelt': -9.166666666667e-06, 'crpix': 421.0}, {'axis': 2, 'ctype': 'DEC--SIN', 'cunit': 'deg', 'naxis': 840, 'crval': 31.22541666667, 'cdelt': 9.166666666667e-06, 'crpix': 421.0}, {'axis': 3, 'ctype': 'FREQ', 'cunit': 'Hz', 'naxis': 478, 'crval': 363908317596.0, 'cdelt': 122189.173584, 'crpix': 1.0}, {'axis': 4, 'ctype': 'STOKES', 'cunit': '', 'naxis': 1, 'crval': 1.0, 'cdelt': 1.0, 'crpix': 1.0}], 'object': 'NGC1333_IRAS4A', 'bunit': 'Jy/beam', 'specsys': 'LSRK', 'restfreq_hz': 363945894000.0, 'restfreq_compatible_with_prereg_target_ghz': True, 'channel_count': 478, 'frequency_low_hz': 363908317596.0, 'frequency_high_hz': 363966601831.79956, 'nominal_radio_velocity_windows_coverage': {'core': True, 'foreground_test': True}, 'velocity_convention_used_for_metadata_check': 'radio; nominal-rest-frequency; no line data inspected'}}, 'independent_arithmetic_from_reported_header': {'reconstructed_high_frequency_hz': 363966601831.79956, 'nominal_radio_frequency_windows_hz': {'core': {'frequency_range_hz': [363937392050.2951, 363938120445.9841], 'covered_by_reported_axis': True}, 'foreground_test': {'frequency_range_hz': [363935813859.6356, 363936542255.3246], 'covered_by_reported_axis': True}}, 'channel_spacing_hz': 122189.173584, 'rest_frequency_delta_to_contract_khz': 4.0}, 'provenance_status': {'readme_X22_previously_verified_in_drive': True, 'continuum_X22_previously_verified_in_drive': True, 'X22_SPW25_file_previously_confirmed_in_drive': True, 'uploaded_report_locally_verified_for_internal_consistency': True, 'original_contract_amendment_adopted': False, 'GVH_USED': False, 'line_amplitudes_not_inspected_by_present_code': True, 'line_test_authorized': False}, 'decision': 'X22_SPW25_REPORTED_METADATA_CONSISTENT__PRESERVE_ABSTAIN_LINE_TEST', 'recommended_next': 'Formal SPW transition mapping review; keep preregistered windows, thresholds and negative controls intact. Optional separate independent archive-level checksum comparison.'}, 'old_headers': [{'member': 'X22', 'name': 'member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw27.cube.I.pbcor.fits', 'relative_path': 'ALMA_2015.1.00147.S/X22/member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw27.cube.I.pbcor.fits', 'size_MB': '1286.6171264648438', 'spw': '27', 'product_class': 'cube_pbcor', 'readable': 'True', 'error': '', 'NAXIS': '4', 'BUNIT': 'Jy/beam', 'OBJECT': 'NGC1333_IRAS4A', 'DATE-OBS': '2016-07-23T13:09:39.264000', 'SPECSYS': 'LSRK', 'BMAJ': '6.856464112613e-05', 'BMIN': '4.705117886315e-05', 'BPA': '-4.746041830652', 'spectral_axis': '3', 'spectral_ctype': 'FREQ', 'spectral_cunit': 'Hz', 'spectral_nchan': '478', 'spectral_crval': '362592851740.0', 'spectral_cdelt': '122188.7150879', 'spectral_crpix': '1.0', 'restfreq_Hz': '362630304000.0', 'approx_dv_kms': '0.10101542764628746'}, {'member': 'X22', 'name': 'member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw29.cube.I.pbcor.fits', 'relative_path': 'ALMA_2015.1.00147.S/X22/member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw29.cube.I.pbcor.fits', 'size_MB': '1286.6171264648438', 'spw': '29', 'product_class': 'cube_pbcor', 'readable': 'True', 'error': '', 'NAXIS': '4', 'BUNIT': 'Jy/beam', 'OBJECT': 'NGC1333_IRAS4A', 'DATE-OBS': '2016-07-23T13:09:39.264000', 'SPECSYS': 'LSRK', 'BMAJ': '6.846077951715e-05', 'BMIN': '4.710876397171e-05', 'BPA': '-4.824687959391', 'spectral_axis': '3', 'spectral_ctype': 'FREQ', 'spectral_cunit': 'Hz', 'spectral_nchan': '478', 'spectral_crval': '362698588244.0', 'spectral_cdelt': '122188.7520752', 'spectral_crpix': '1.0', 'restfreq_Hz': '362736048000.0', 'approx_dv_kms': '0.10098601042424327'}, {'member': 'X22', 'name': 'member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw31.cube.I.pbcor.fits', 'relative_path': 'ALMA_2015.1.00147.S/X22/member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw31.cube.I.pbcor.fits', 'size_MB': '2578.6093139648438', 'spw': '31', 'product_class': 'cube_pbcor', 'readable': 'True', 'error': '', 'NAXIS': '4', 'BUNIT': 'Jy/beam', 'OBJECT': 'NGC1333_IRAS4A', 'DATE-OBS': '2016-07-23T13:09:39.264000', 'SPECSYS': 'LSRK', 'BMAJ': '6.80262143508e-05', 'BMIN': '4.674052169928e-05', 'BPA': '-6.25715546687', 'spectral_axis': '3', 'spectral_ctype': 'FREQ', 'spectral_cunit': 'Hz', 'spectral_nchan': '958', 'spectral_crval': '364208311370.0', 'spectral_cdelt': '122125.8200073', 'spectral_crpix': '1.0', 'restfreq_Hz': '364275141000.0', 'approx_dv_kms': '0.10050754400848354'}, {'member': 'X22', 'name': 'member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw33.cube.I.pbcor.fits', 'relative_path': 'ALMA_2015.1.00147.S/X22/member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw33.cube.I.pbcor.fits', 'size_MB': '1286.6171264648438', 'spw': '33', 'product_class': 'cube_pbcor', 'readable': 'True', 'error': '', 'NAXIS': '4', 'BUNIT': 'Jy/beam', 'OBJECT': 'NGC1333_IRAS4A', 'DATE-OBS': '2016-07-23T13:09:39.264000', 'SPECSYS': 'LSRK', 'BMAJ': '6.796436009331e-05', 'BMIN': '4.631449916506e-05', 'BPA': '-6.580501578654', 'spectral_axis': '3', 'spectral_ctype': 'FREQ', 'spectral_cunit': 'Hz', 'spectral_nchan': '478', 'spectral_crval': '364065742650.0', 'spectral_cdelt': '122189.2283936', 'spectral_crpix': '1.0', 'restfreq_Hz': '364103249000.0', 'approx_dv_kms': '0.10060720200066312'}, {'member': 'X22', 'name': 'member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw35.cube.I.pbcor.fits', 'relative_path': 'ALMA_2015.1.00147.S/X22/member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw35.cube.I.pbcor.fits', 'size_MB': '2578.6093139648438', 'spw': '35', 'product_class': 'cube_pbcor', 'readable': 'True', 'error': '', 'NAXIS': '4', 'BUNIT': 'Jy/beam', 'OBJECT': 'NGC1333_IRAS4A', 'DATE-OBS': '2016-07-23T13:09:39.264000', 'SPECSYS': 'LSRK', 'BMAJ': '6.973497584779e-05', 'BMIN': '4.768089977865e-05', 'BPA': '-7.101238024543', 'spectral_axis': '3', 'spectral_ctype': 'FREQ', 'spectral_cunit': 'Hz', 'spectral_nchan': '958', 'spectral_crval': '351818987273.5', 'spectral_cdelt': '-122123.6410522', 'spectral_crpix': '1.0', 'restfreq_Hz': '351768645000.0', 'approx_dv_kms': '0.10407905039674226'}, {'member': 'X22', 'name': 'member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw37.cube.I.pbcor.fits', 'relative_path': 'ALMA_2015.1.00147.S/X22/member.uid___A001_X2f6_X22.NGC1333_IRAS4A_sci.spw37.cube.I.pbcor.fits', 'size_MB': '5162.593688964844', 'spw': '37', 'product_class': 'cube_pbcor', 'readable': 'True', 'error': '', 'NAXIS': '4', 'BUNIT': 'Jy/beam', 'OBJECT': '', 'DATE-OBS': '2016-07-23T13:09:39.264000', 'SPECSYS': 'LSRK', 'BMAJ': '6.973635042976e-05', 'BMIN': '4.790335611656e-05', 'BPA': '-7.445972597404', 'spectral_axis': '3', 'spectral_ctype': 'FREQ', 'spectral_cunit': 'Hz', 'spectral_nchan': '1918', 'spectral_crval': '351627934170.4', 'spectral_cdelt': '-976529.3659058', 'spectral_crpix': '1.0', 'restfreq_Hz': '350700000000.0', 'approx_dv_kms': '0.8347765580669552'}, {'member': 'X24', 'name': 'member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw25_27_29_31_33_35_37.cont.I.pbcor.fits', 'relative_path': 'ALMA_2015.1.00147.S/X24/member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw25_27_29_31_33_35_37.cont.I.pbcor.fits', 'size_MB': '0.4010009765625', 'spw': '25', 'product_class': 'continuum_pbcor', 'readable': 'True', 'error': '', 'NAXIS': '4', 'BUNIT': 'Jy/beam', 'OBJECT': 'NGC1333_IRAS4A', 'DATE-OBS': '2016-12-14T02:46:23.664000', 'SPECSYS': 'LSRK', 'BMAJ': '0.0001822466982736', 'BMIN': '0.0001295142372449', 'BPA': '-11.7789182663', 'spectral_axis': '3', 'spectral_ctype': 'FREQ', 'spectral_cunit': 'Hz', 'spectral_nchan': '1', 'spectral_crval': '357454591074.3', 'spectral_cdelt': '13740200125.35', 'spectral_crpix': '1.0', 'restfreq_Hz': '363945894000.0', 'approx_dv_kms': '11318.188876148124'}, {'member': 'X24', 'name': 'member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw27.cube.I.pbcor.fits', 'relative_path': 'ALMA_2015.1.00147.S/X24/member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw27.cube.I.pbcor.fits', 'size_MB': '186.7291259765625', 'spw': '27', 'product_class': 'cube_pbcor', 'readable': 'True', 'error': '', 'NAXIS': '4', 'BUNIT': 'Jy/beam', 'OBJECT': 'NGC1333_IRAS4A', 'DATE-OBS': '2016-12-14T02:46:23.664000', 'SPECSYS': 'LSRK', 'BMAJ': '0.0001817799357469', 'BMIN': '0.0001328116584031', 'BPA': '-10.90097262006', 'spectral_axis': '3', 'spectral_ctype': 'FREQ', 'spectral_cunit': 'Hz', 'spectral_nchan': '478', 'spectral_crval': '362592793678.0', 'spectral_cdelt': '122243.9969482', 'spectral_crpix': '1.0', 'restfreq_Hz': '362630304000.0', 'approx_dv_kms': '0.10106113007269622'}, {'member': 'X24', 'name': 'member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw29.cube.I.pbcor.fits', 'relative_path': 'ALMA_2015.1.00147.S/X24/member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw29.cube.I.pbcor.fits', 'size_MB': '186.7291259765625', 'spw': '29', 'product_class': 'cube_pbcor', 'readable': 'True', 'error': '', 'NAXIS': '4', 'BUNIT': 'Jy/beam', 'OBJECT': 'NGC1333_IRAS4A', 'DATE-OBS': '2016-12-14T02:46:23.664000', 'SPECSYS': 'LSRK', 'BMAJ': '0.000182295602544', 'BMIN': '0.0001320104605282', 'BPA': '-11.06995925382', 'spectral_axis': '3', 'spectral_ctype': 'FREQ', 'spectral_cunit': 'Hz', 'spectral_nchan': '478', 'spectral_crval': '362698543747.0', 'spectral_cdelt': '122244.0454102', 'spectral_crpix': '1.0', 'restfreq_Hz': '362736048000.0', 'approx_dv_kms': '0.10103170901114156'}, {'member': 'X24', 'name': 'member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw31.cube.I.pbcor.fits', 'relative_path': 'ALMA_2015.1.00147.S/X24/member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw31.cube.I.pbcor.fits', 'size_MB': '374.227294921875', 'spw': '31', 'product_class': 'cube_pbcor', 'readable': 'True', 'error': '', 'NAXIS': '4', 'BUNIT': 'Jy/beam', 'OBJECT': 'NGC1333_IRAS4A', 'DATE-OBS': '2016-12-14T02:46:23.664000', 'SPECSYS': 'LSRK', 'BMAJ': '0.0001815466600835', 'BMIN': '0.0001317708042756', 'BPA': '-11.26107002829', 'spectral_axis': '3', 'spectral_ctype': 'FREQ', 'spectral_cunit': 'Hz', 'spectral_nchan': '958', 'spectral_crval': '364208331920.0', 'spectral_cdelt': '122161.3826904', 'spectral_crpix': '1.0', 'restfreq_Hz': '364275141000.0', 'approx_dv_kms': '0.10053681151257497'}, {'member': 'X24', 'name': 'member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw33.cube.I.pbcor.fits', 'relative_path': 'ALMA_2015.1.00147.S/X24/member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw33.cube.I.pbcor.fits', 'size_MB': '186.7291259765625', 'spw': '33', 'product_class': 'cube_pbcor', 'readable': 'True', 'error': '', 'NAXIS': '4', 'BUNIT': 'Jy/beam', 'OBJECT': 'NGC1333_IRAS4A', 'DATE-OBS': '2016-12-14T02:46:23.664000', 'SPECSYS': 'LSRK', 'BMAJ': '0.0001822682026438', 'BMIN': '0.000132064489057', 'BPA': '-11.19207717656', 'spectral_axis': '3', 'spectral_ctype': 'FREQ', 'spectral_cunit': 'Hz', 'spectral_nchan': '478', 'spectral_crval': '364065744938.0', 'spectral_cdelt': '122244.6712036', 'spectral_crpix': '1.0', 'restfreq_Hz': '364103249000.0', 'approx_dv_kms': '0.10065285206375366'}, {'member': 'X24', 'name': 'member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw35.cube.I.pbcor.fits', 'relative_path': 'ALMA_2015.1.00147.S/X24/member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw35.cube.I.pbcor.fits', 'size_MB': '374.227294921875', 'spw': '35', 'product_class': 'cube_pbcor', 'readable': 'True', 'error': '', 'NAXIS': '4', 'BUNIT': 'Jy/beam', 'OBJECT': 'NGC1333_IRAS4A', 'DATE-OBS': '2016-12-14T02:46:23.664000', 'SPECSYS': 'LSRK', 'BMAJ': '0.0001871778532103', 'BMIN': '0.0001351585748651', 'BPA': '-10.79480839118', 'spectral_axis': '3', 'spectral_ctype': 'FREQ', 'spectral_cunit': 'Hz', 'spectral_nchan': '958', 'spectral_crval': '351819031532.1', 'spectral_cdelt': '-122158.5205078', 'spectral_crpix': '1.0', 'restfreq_Hz': '351768645000.0', 'approx_dv_kms': '0.10410877617781075'}, {'member': 'X24', 'name': 'member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw37.cube.I.pbcor.fits', 'relative_path': 'ALMA_2015.1.00147.S/X24/member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw37.cube.I.pbcor.fits', 'size_MB': '749.2291259765625', 'spw': '37', 'product_class': 'cube_pbcor', 'readable': 'True', 'error': '', 'NAXIS': '4', 'BUNIT': 'Jy/beam', 'OBJECT': 'NGC1333_IRAS4A', 'DATE-OBS': '2016-12-14T02:46:23.664000', 'SPECSYS': 'LSRK', 'BMAJ': '0.0001898198256084', 'BMIN': '0.0001350553868477', 'BPA': '-11.47462671879', 'spectral_axis': '3', 'spectral_ctype': 'FREQ', 'spectral_cunit': 'Hz', 'spectral_nchan': '1918', 'spectral_crval': '351628090969.3', 'spectral_cdelt': '-976664.2546997', 'spectral_crpix': '1.0', 'restfreq_Hz': '350700000000.0', 'approx_dv_kms': '0.8348918664304567'}], 'old_mapping': [{'member': 'X22', 'original_spw': '27', 'label_from_frozen_contract': 'H2CO', 'target_ghz': '351.76864', 'rest_candidate_spw': '35.0', 'header_rest_difference_khz': '5.0', 'coverage_core': 'True', 'coverage_foreground': 'True', 'frequency_covered': 'True', 'header_rest_match_1MHz': 'True', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}, {'member': 'X24', 'original_spw': '27', 'label_from_frozen_contract': 'H2CO', 'target_ghz': '351.76864', 'rest_candidate_spw': '35.0', 'header_rest_difference_khz': '5.0', 'coverage_core': 'True', 'coverage_foreground': 'True', 'frequency_covered': 'True', 'header_rest_match_1MHz': 'True', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}, {'member': 'X22', 'original_spw': '29', 'label_from_frozen_contract': 'HNC', 'target_ghz': '362.6303', 'rest_candidate_spw': '27.0', 'header_rest_difference_khz': '4.0', 'coverage_core': 'True', 'coverage_foreground': 'True', 'frequency_covered': 'True', 'header_rest_match_1MHz': 'True', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}, {'member': 'X24', 'original_spw': '29', 'label_from_frozen_contract': 'HNC', 'target_ghz': '362.6303', 'rest_candidate_spw': '27.0', 'header_rest_difference_khz': '4.0', 'coverage_core': 'True', 'coverage_foreground': 'True', 'frequency_covered': 'True', 'header_rest_match_1MHz': 'True', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}, {'member': 'X22', 'original_spw': '31', 'label_from_frozen_contract': 'H2CO', 'target_ghz': '362.73605', 'rest_candidate_spw': '29.0', 'header_rest_difference_khz': '2.0', 'coverage_core': 'True', 'coverage_foreground': 'True', 'frequency_covered': 'True', 'header_rest_match_1MHz': 'True', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}, {'member': 'X24', 'original_spw': '31', 'label_from_frozen_contract': 'H2CO', 'target_ghz': '362.73605', 'rest_candidate_spw': '29.0', 'header_rest_difference_khz': '2.0', 'coverage_core': 'True', 'coverage_foreground': 'True', 'frequency_covered': 'True', 'header_rest_match_1MHz': 'True', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}, {'member': 'X22', 'original_spw': '33', 'label_from_frozen_contract': 'H2CO', 'target_ghz': '363.94589', 'rest_candidate_spw': '', 'header_rest_difference_khz': '', 'coverage_core': 'False', 'coverage_foreground': 'False', 'frequency_covered': 'False', 'header_rest_match_1MHz': 'False', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}, {'member': 'X24', 'original_spw': '33', 'label_from_frozen_contract': 'H2CO', 'target_ghz': '363.94589', 'rest_candidate_spw': '25.0', 'header_rest_difference_khz': '4.0', 'coverage_core': 'True', 'coverage_foreground': 'True', 'frequency_covered': 'True', 'header_rest_match_1MHz': 'True', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}, {'member': 'X22', 'original_spw': '35', 'label_from_frozen_contract': 'H2CO', 'target_ghz': '364.10325', 'rest_candidate_spw': '33.0', 'header_rest_difference_khz': '1.0', 'coverage_core': 'True', 'coverage_foreground': 'True', 'frequency_covered': 'True', 'header_rest_match_1MHz': 'True', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}, {'member': 'X24', 'original_spw': '35', 'label_from_frozen_contract': 'H2CO', 'target_ghz': '364.10325', 'rest_candidate_spw': '33.0', 'header_rest_difference_khz': '1.0', 'coverage_core': 'True', 'coverage_foreground': 'True', 'frequency_covered': 'True', 'header_rest_match_1MHz': 'True', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}, {'member': 'X22', 'original_spw': '37', 'label_from_frozen_contract': 'H2CO', 'target_ghz': '364.27514', 'rest_candidate_spw': '31.0', 'header_rest_difference_khz': '1.0', 'coverage_core': 'True', 'coverage_foreground': 'True', 'frequency_covered': 'True', 'header_rest_match_1MHz': 'True', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}, {'member': 'X24', 'original_spw': '37', 'label_from_frozen_contract': 'H2CO', 'target_ghz': '364.27514', 'rest_candidate_spw': '31.0', 'header_rest_difference_khz': '1.0', 'coverage_core': 'True', 'coverage_foreground': 'True', 'frequency_covered': 'True', 'header_rest_match_1MHz': 'True', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}, {'member': 'X22', 'original_spw': '37', 'label_from_frozen_contract': 'H2CO', 'target_ghz': '364.28888', 'rest_candidate_spw': '31.0', 'header_rest_difference_khz': '13739.0', 'coverage_core': 'True', 'coverage_foreground': 'True', 'frequency_covered': 'True', 'header_rest_match_1MHz': 'False', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}, {'member': 'X24', 'original_spw': '37', 'label_from_frozen_contract': 'H2CO', 'target_ghz': '364.28888', 'rest_candidate_spw': '31.0', 'header_rest_difference_khz': '13739.0', 'coverage_core': 'True', 'coverage_foreground': 'True', 'frequency_covered': 'True', 'header_rest_match_1MHz': 'False', 'original_assignment_confirmed': 'False', 'other_matching_spws': '[]'}], 'readme_first_40_lines': ['Atacama Large Millimeter/submillimeter Array (ALMA)', '', '#####', '', 'Cycle: 3', 'Project code: 2015.1.00147.S', 'SB name: NGC1333__a_07_TE', 'PI name: Yu-Nung Su (Username: ynsu)', 'Project title: Probing the Inward Motion in the Class 0 Protostar NGC 1333 IRAS 4A', 'Configuration: TWELVE-M', 'Proposed rms: 7.0 mJy per 0.15 km/s (0.176 MHz)', 'Proposed beam size: 0.2 arcsec', 'CASA version used for reduction: 4.7.0', 'QA2 Result: PASS', 'Total Number of Member SBs in this OUS Group: 2', 'Comments from Reducer:', '', 'Pipeline calibrated and imaged dataset.', '', 'For the representative spw 35 (uid___A001_X2f6_X22.s31_0.NGC1333_IRAS4A_sci.spw35.cube.I.iter1.image):', '', 'Beam size: 0.25 * 0.17 arcsec', 'RMS: 7.7 mJy per 0.1221 MHz (native resolution)', '', 'The requested RMS for 0.1221 MHz calculated from the OT sensitivity calculator is ~8.3 mJy per 0.1221 MHz. This is QA2 PASS.', '', '', '', '#####', '', 'README OVERVIEW', '', ' Introduction', ' Total Power Data', '   Pipeline-calibrated TP data', '   Manually calibrated TP data', ' Primary Beam Correction', ' How to restore the calibrated MeasurementSet for your data', '', '#####'], 'x24_spw25_pinned_header': {'object': 'NGC1333_IRAS4A', 'bunit': 'Jy/beam', 'specsys': 'LSRK', 'naxis': 4, 'axes': [{'axis': 1, 'ctype': 'RA---SIN', 'unit': 'deg', 'n': 320, 'crval': 52.29375, 'cdelt': -2.472222222222e-05, 'crpix': 161.0}, {'axis': 2, 'ctype': 'DEC--SIN', 'unit': 'deg', 'n': 320, 'crval': 31.22541666667, 'cdelt': 2.472222222222e-05, 'crpix': 161.0}, {'axis': 3, 'ctype': 'FREQ', 'unit': 'Hz', 'n': 478, 'crval': 363908428308.0, 'cdelt': 122244.5992432, 'crpix': 1.0}, {'axis': 4, 'ctype': 'STOKES', 'unit': '', 'n': 1, 'crval': 1.0, 'cdelt': 1.0, 'crpix': 1.0}], 'restfreq_hz': 363945894000.0, 'bmaj_arcsec': 0.6555126108744, 'bmin_arcsec': 0.47203996120992003}}
DRIVE_INVENTORY_SNAPSHOT = {'retrieval_method': 'Connected Google Drive folder fetch, direct child listing; 2026-11-20', 'project_code': '2015.1.00147.S', 'folders': {'X22': {'id': '1fVUzR6djAh2t5WvayOV_mHjofbOJyNgc', 'n_files': 8, 'primary_spws': [27, 29, 31, 33, 35, 37], 'has_continuum': True, 'has_spw25_cube': True}, 'X24': {'id': '1wcVoPE1ysXoSb7ZbQgUKu4oB8Op3BMn9', 'n_files': 59, 'primary_spws': [27, 29, 31, 33, 35, 37], 'has_continuum': True, 'has_spw25_cube': True}, 'README': {'id': '1rW4uSOEiydsEp_jHHCZTS5r_r1My-UMo', 'n_files': 1, 'X22_readme_present': True, 'X24_readme_present': False}}, 'named_files': {'X22_continuum': {'id': '1N-befCM6OPlG9jA8iiQE1I8oQgNOmDDw', 'size': 2831040}, 'X24_continuum': {'id': '1Ud0wGIwNdzac-hSqWArlBS1-Alf0Ky-v', 'size': 420480}, 'X22_spw25_cube': {'id': '1D6kE5LEd524imWNgRy0FZ59SfXGHlYmY', 'size': 1349115840}, 'X24_spw25_cube': {'id': '1bvAypfGrOdt9ToBcO4imx0n8jPAd4yS0', 'size': 195799680}, 'X22_README': {'id': '1vKEWqNRr1wttRwYAxm5L3tOVhGDca1uN', 'size': 19651}}}
FROZEN_CONTRACT = SOURCE_SNAPSHOT['contract']
H0_STATUS = SOURCE_SNAPSHOT['h0_status']
UPSTREAM_R1 = SOURCE_SNAPSHOT['r1_review']
HEADERS = SOURCE_SNAPSHOT['old_headers']
print('Source snapshot loaded; original preregistration SHA-256:',EXPECTED_SOURCE_SHA['contract'])


Source snapshot loaded; original preregistration SHA-256: ec019b74dfec41fba05ccc78170a98324386e02cdf5f9f2267beb4f145a54831


In [2]:
# Source exactness; source snapshots never overwritten. These are small files only.
source_audit={}
for key,name in INPUT_FILENAMES.items():
    p=BASE/name
    if p.exists():
        actual=sha256_file(p)
        gate('source_sha_'+key,actual==EXPECTED_SOURCE_SHA[key],name)
        source_audit[key]={'available':True,'sha256':actual,'representation':'DIRECT_LOCAL_FILE'}
    else:
        source_audit[key]={'available':False,'sha256':EXPECTED_SOURCE_SHA[key],'representation':'EMBEDDED_PINNED_SOURCE_ONLY'}

gate('frozen_source_project',FROZEN_CONTRACT['project_code']=='2015.1.00147.S')
gate('frozen_members',FROZEN_CONTRACT['members']==['X22','X24'])
gate('frozen_velocities',FROZEN_CONTRACT['velocity_windows_kms']=={'core':[6.4,7.0],'foreground_test':[7.7,8.3]})
gate('frozen_original_spws',FROZEN_CONTRACT['primary_spws']==[27,29,31,33,35,37])
gate('frozen_spw25_not_primary',FROZEN_CONTRACT['spw25_primary_test'] is False)
gate('original_h0_failed',H0_STATUS['PATCH10H0_PASS'] is False)
gate('original_h0_not_authorized',H0_STATUS['NEXT_PATCH_AUTHORIZED'] is False)
gate('metadata_only_flag',not LINE_AMPLITUDES_INSPECTED and not GVH_USED and not ALLOW_VELOCITY_WINDOW_RETUNING)

archive_recovery={
 'X22_readme_present':DRIVE_INVENTORY_SNAPSHOT['folders']['README']['X22_readme_present'],
 'X22_continuum_present':DRIVE_INVENTORY_SNAPSHOT['folders']['X22']['has_continuum'],
 'X24_continuum_present':DRIVE_INVENTORY_SNAPSHOT['folders']['X24']['has_continuum'],
 'X22_spw25_cube_present':DRIVE_INVENTORY_SNAPSHOT['folders']['X22']['has_spw25_cube'],
 'X24_spw25_cube_present':DRIVE_INVENTORY_SNAPSHOT['folders']['X24']['has_spw25_cube'],
 'X24_readme_present':DRIVE_INVENTORY_SNAPSHOT['folders']['README']['X24_readme_present']
}
gate('drive_recovery_targeted_items_present',all(archive_recovery[x] for x in archive_recovery if x!='X24_readme_present'))
gate('readme_x22_qa2_pass','QA2 Result: PASS' in '\n'.join(SOURCE_SNAPSHOT['readme_first_40_lines']))
gate('readme_x22_project','2015.1.00147.S' in '\n'.join(SOURCE_SNAPSHOT['readme_first_40_lines']))
print('Archive recovery:',json.dumps(archive_recovery,indent=2))


PASS source_sha_contract
PASS source_sha_h0_status
PASS source_sha_old_headers
PASS source_sha_old_hashes
PASS source_sha_old_mapping
PASS source_sha_r1_review
PASS source_sha_readme
PASS source_sha_cont22
PASS source_sha_cont24
PASS source_sha_spw25x24
PASS frozen_source_project
PASS frozen_members
PASS frozen_velocities
PASS frozen_original_spws
PASS frozen_spw25_not_primary
PASS original_h0_failed
PASS original_h0_not_authorized
PASS metadata_only_flag
PASS drive_recovery_targeted_items_present
PASS readme_x22_qa2_pass
PASS readme_x22_project
Archive recovery: {
  "X22_readme_present": true,
  "X22_continuum_present": true,
  "X24_continuum_present": true,
  "X22_spw25_cube_present": true,
  "X24_spw25_cube_present": true,
  "X24_readme_present": false
}


## FITS header-only checks

Read the primary FITS header cards of small files without loading spectral amplitudes. For the two *continuum* images, full small-file SHA-256 hashes are computed. We do **not** open any line-cube data arrays.

In [3]:
def parse_fits_header(path):
    out={}
    with open(path,'rb') as fh:
        while True:
            block=fh.read(2880)
            assert len(block)==2880, f'Truncated header {path}'
            for j in range(0,2880,80):
                line=block[j:j+80].decode('ascii',errors='strict')
                key=line[:8].strip()
                if key=='END':return out
                if not key or line[8:10]!='= ':continue
                v=line[10:].strip()
                if v.startswith("'"):
                    end=v.find("'",1)
                    out[key]=v[1:end].strip()
                else:
                    core=v.split('/',1)[0].strip()
                    if core in ('T','F'): out[key]=core=='T'
                    else:
                        try:
                            out[key]=float(core.replace('D','E')) if any(x in core.upper() for x in ('.','E','D')) else int(core)
                        except ValueError:out[key]=core

def audit_header(h):
    axes=[]
    for i in range(1,int(h.get('NAXIS',0))+1):
        axes.append({'axis':i,'ctype':h.get(f'CTYPE{i}'),'unit':h.get(f'CUNIT{i}'), 'n':int(h.get(f'NAXIS{i}',0)), 'crval':h.get(f'CRVAL{i}'),'cdelt':h.get(f'CDELT{i}'),'crpix':h.get(f'CRPIX{i}')})
    return {'object':h.get('OBJECT'),'bunit':h.get('BUNIT'),'specsys':h.get('SPECSYS'),'naxis':h.get('NAXIS'),'axes':axes,'restfreq_hz':h.get('RESTFRQ',h.get('RESTFREQ')),'bmaj_arcsec':3600*float(h['BMAJ']),'bmin_arcsec':3600*float(h['BMIN'])}

SMALL_NAMES=['cont22','cont24','spw25x24']
small_header={}
small_sha={}
for x in SMALL_NAMES:
    f=BASE/INPUT_FILENAMES[x]
    if not f.exists():
        print('SKIP local raw header:',x,'(input not mounted here)');continue
    hd=parse_fits_header(f)
    small_header[x]=audit_header(hd)
    # Byte-level checksum was already verified against the source SHA in the source cell.
    small_sha[x]=source_audit[x]['sha256']
    gate('direct_fits_valid_'+x,hd.get('SIMPLE') is True and hd.get('NAXIS')==4 and hd.get('BITPIX')==-32)
    gate('direct_fits_object_'+x,hd.get('OBJECT')=='NGC1333_IRAS4A' and hd.get('BUNIT')=='Jy/beam')
    gate('direct_fits_rest_363945894_'+x,hd.get('RESTFRQ')==363945894000.0)
    gate('direct_fits_spatial_axis_'+x,hd.get('CTYPE1')=='RA---SIN' and hd.get('CTYPE2')=='DEC--SIN')

if {'cont22','cont24'}.issubset(small_header):
    h22=small_header['cont22'];h24=small_header['cont24']
    gate('continuum_x22_dimensions',[h22['axes'][0]['n'],h22['axes'][1]['n']]==[840,840])
    gate('continuum_x24_dimensions',[h24['axes'][0]['n'],h24['axes'][1]['n']]==[320,320])
    gate('continuum_spatial_centers_match',abs(h22['axes'][0]['crval']-h24['axes'][0]['crval'])<1e-10 and abs(h22['axes'][1]['crval']-h24['axes'][1]['crval'])<1e-10)
    gate('continuum_beam_order',h22['bmaj_arcsec']<h24['bmaj_arcsec'])
print('Continuum header metadata:',json.dumps({k:v for k,v in small_header.items() if k.startswith('cont')},indent=2))


PASS direct_fits_valid_cont22
PASS direct_fits_object_cont22
PASS direct_fits_rest_363945894_cont22
PASS direct_fits_spatial_axis_cont22
PASS direct_fits_valid_cont24
PASS direct_fits_object_cont24
PASS direct_fits_rest_363945894_cont24
PASS direct_fits_spatial_axis_cont24
PASS direct_fits_valid_spw25x24
PASS direct_fits_object_spw25x24
PASS direct_fits_rest_363945894_spw25x24
PASS direct_fits_spatial_axis_spw25x24
PASS continuum_x22_dimensions
PASS continuum_x24_dimensions
PASS continuum_spatial_centers_match
PASS continuum_beam_order
Continuum header metadata: {
  "cont22": {
    "object": "NGC1333_IRAS4A",
    "bunit": "Jy/beam",
    "specsys": "LSRK",
    "naxis": 4,
    "axes": [
      {
        "axis": 1,
        "ctype": "RA---SIN",
        "unit": "deg",
        "n": 840,
        "crval": 52.29375,
        "cdelt": -9.166666666667e-06,
        "crpix": 421.0
      },
      {
        "axis": 2,
        "ctype": "DEC--SIN",
        "unit": "deg",
        "n": 840,
        "crval"

In [4]:
# User's Colab run: FITS header and SHA *reported*, not an independently re-read 1.349 GB cube.
r1=UPSTREAM_R1['reported']
x22_25=r1['fits_header']; x22_sha=r1['sha256_from_colab_output']
gate('r1_reported_cube_name',UPSTREAM_R1['source_file'].endswith('(2).ipynb'))
gate('r1_reported_size',r1['file_size_bytes']==DRIVE_INVENTORY_SNAPSHOT['named_files']['X22_spw25_cube']['size'])
gate('r1_reported_sha_format',bool(re.fullmatch('[0-9a-f]{64}',x22_sha)))
gate('r1_no_line_intensity_access',UPSTREAM_R1['provenance_status']['line_amplitudes_not_inspected_by_present_code'])
gate('r1_reported_rest',x22_25['restfreq_hz']==363945894000.0)
gate('r1_reported_nchannels',x22_25['channel_count']==478)
gate('r1_reported_spectral_frame',x22_25['specsys']=='LSRK')
gate('r1_reconstructed_axis',abs(x22_25['frequency_high_hz']-(x22_25['axes'][2]['crval']+(x22_25['axes'][2]['naxis']-1)*x22_25['axes'][2]['cdelt']))<1.0)

if 'spw25x24' not in small_header and SOURCE_SNAPSHOT['x24_spw25_pinned_header'] is not None:
    small_header['spw25x24']=SOURCE_SNAPSHOT['x24_spw25_pinned_header']
    print('X24 SPW25: archived direct header snapshot only (raw file not available in this runtime)')
gate('x24_spw25_cube_478',small_header.get('spw25x24',{}).get('axes',[{}, {}, {'n':None}])[2]['n']==478)
gate('x24_direct_spw25_rest',small_header.get('spw25x24',{}).get('restfreq_hz')==x22_25['restfreq_hz'])
# Availability of X22 independent raw-byte re-read is NOT promoted by checking the Colab output.
x22_spw25_evidence='USER_COLAB_REPORTED_FITS_HEADER_AND_SHA256__DRIVE_FILE_INVENTORY_CONFIRMED__NOT_INDEPENDENTLY_HASHED_THIS_RUN'
print('X22 SPW25:',x22_spw25_evidence,'SHA256 reported=',x22_sha)


PASS r1_reported_cube_name
PASS r1_reported_size
PASS r1_reported_sha_format
PASS r1_no_line_intensity_access
PASS r1_reported_rest
PASS r1_reported_nchannels
PASS r1_reported_spectral_frame
PASS r1_reconstructed_axis
PASS x24_spw25_cube_478
PASS x24_direct_spw25_rest
X22 SPW25: USER_COLAB_REPORTED_FITS_HEADER_AND_SHA256__DRIVE_FILE_INVENTORY_CONFIRMED__NOT_INDEPENDENTLY_HASHED_THIS_RUN SHA256 reported= 67fd0dee3db8f9ba97db1f63d0eb655582065d1f6f57f16e1d046a3eae728a78


## Immutable cross-member SPW reconciliation

Original `expected_spw_mapping` is the *target list*, not truth about actual SPW labels. For each target in each member, find a cube whose `RESTFRQ` is within **1 MHz** of the nominal target, otherwise flag mere **frequency-axis inclusion** without claiming line identification. The second 364.28888 GHz doublet component must be distinguished from its header `RESTFRQ` counterpart 364.275141 GHz.

In [5]:
# Assemble archival primary headers + new SPW25 metadata without ever reading a spectral intensity sample.
spw_rows={}
for row in HEADERS:
    if row['product_class']!='cube_pbcor':continue
    mem=row['member']; spw=int(row['spw'])
    f0=float(row['spectral_crval'])+(1.-float(row['spectral_crpix']))*float(row['spectral_cdelt'])
    fend=float(row['spectral_crval'])+(int(row['spectral_nchan'])-float(row['spectral_crpix']))*float(row['spectral_cdelt'])
    spw_rows[(mem,spw)]={'restfreq_hz':float(row['restfreq_Hz']),'f_low':min(f0,fend),'f_high':max(f0,fend),'nchan':int(row['spectral_nchan']),'evidence':'ARCHIVED_H0_FITS_HEADER_AUDIT'}

spw_rows[('X22',25)]={'restfreq_hz':float(x22_25['restfreq_hz']),'f_low':x22_25['frequency_low_hz'],'f_high':x22_25['frequency_high_hz'],'nchan':x22_25['channel_count'],'evidence':x22_spw25_evidence}
ax=small_header['spw25x24']['axes'][2]
fstart=float(ax['crval'])+(1-float(ax['crpix']))*float(ax['cdelt'])
fend=float(ax['crval'])+(ax['n']-float(ax['crpix']))*float(ax['cdelt'])
spw_rows[('X24',25)]={'restfreq_hz':float(small_header['spw25x24']['restfreq_hz']),'f_low':min(fstart,fend),'f_high':max(fstart,fend),'nchan':ax['n'],'evidence':'DIRECT_LOCAL_FITS_HEADER_NO_INTENSITY' if source_audit['spw25x24']['available'] else 'PINNED_X24_FITS_HEADER_ARCHIVE_ONLY'}

gate('spectral_cube_inventory_14_entries',len(spw_rows)==14)
gate('spectral_member_spws_complete',all((m,spw) in spw_rows for m in ['X22','X24'] for spw in [25,27,29,31,33,35,37]))

def contains_windows(q,nominal_hz):
    out={}
    for key,window in FROZEN_CONTRACT['velocity_windows_kms'].items():
        freqs=sorted([nominal_hz*(1-v/C_KM_S) for v in window])
        out[key]={'frequencies_hz':freqs,'in_axis':q['f_low']<=freqs[0] and freqs[1]<=q['f_high']}
    return out

row_out=[]
for m in ['X22','X24']:
    for original_spw in FROZEN_CONTRACT['primary_spws']:
        entry=FROZEN_CONTRACT['expected_spw_mapping'][str(original_spw)]
        for target_ghz in entry['rest_GHz_expected']:
            target_hz=float(target_ghz)*1e9
            entries=[(spw,q) for (member,spw),q in spw_rows.items() if member==m]
            rest_matches=sorted([(spw,q) for spw,q in entries if abs(q['restfreq_hz']-target_hz)<=1e6],key=lambda sq:abs(sq[1]['restfreq_hz']-target_hz))
            axis_matches=sorted([(spw,q) for spw,q in entries if q['f_low']<=target_hz<=q['f_high']],key=lambda sq:abs(sq[1]['restfreq_hz']-target_hz))
            chosen=(rest_matches or axis_matches)
            spw,q=chosen[0] if chosen else (None,None)
            w=contains_windows(q,target_hz) if q else {}
            matching_quality='HEADER_REST_MATCH' if rest_matches else ('AXIS_COVERAGE_ONLY' if axis_matches else 'NOT_FOUND')
            row_out.append({
                'member':m,'original_frozen_spw':original_spw,'label_in_original_contract':entry['label'],
                'target_GHz':float(target_ghz),'candidate_spw':spw,'evidence_grade':q['evidence'] if q else 'NONE',
                'match_type':matching_quality,'rest_offset_kHz':round((q['restfreq_hz']-target_hz)/1e3,4) if q else None,
                'frequency_in_axis':bool(q),
                'core_window_covered':bool(w.get('core',{}).get('in_axis',False)),
                'foreground_window_covered':bool(w.get('foreground_test',{}).get('in_axis',False)),
                'original_spw_still_matches':bool(spw==original_spw),
                'chemical_identification_independent':False
            })

coverage={m:sum(r['frequency_in_axis'] and r['core_window_covered'] and r['foreground_window_covered'] for r in row_out if r['member']==m) for m in ['X22','X24']}
gate('two_members_seven_targets_each',len(row_out)==14 and all(sum(r['member']==m for r in row_out)==7 for m in ['X22','X24']))
gate('x22_seven_of_seven_metadata_coverage',coverage['X22']==7)
gate('x24_seven_of_seven_metadata_coverage',coverage['X24']==7)
gate('historical_spw_all_reassigned',all(not r['original_spw_still_matches'] for r in row_out))
gate('x22_36394589_uses_spw25',next(r for r in row_out if r['member']=='X22' and abs(r['target_GHz']-363.94589)<1e-7)['candidate_spw']==25)
gate('x24_36394589_uses_spw25',next(r for r in row_out if r['member']=='X24' and abs(r['target_GHz']-363.94589)<1e-7)['candidate_spw']==25)
gate('doublet_second_axis_only',all(r['match_type']=='AXIS_COVERAGE_ONLY' and r['candidate_spw']==31 for r in row_out if abs(r['target_GHz']-364.28888)<1e-7))
gate('all_matched_windows_covered',all(r['core_window_covered'] and r['foreground_window_covered'] for r in row_out))
gate('species_not_independently_certified',all(r['chemical_identification_independent'] is False for r in row_out))
print('COVERAGE',coverage)
for row in row_out: print(f"{row['member']} {row['target_GHz']:.5f} GHz: frozen SPW{row['original_frozen_spw']} -> header SPW{row['candidate_spw']} ({row['match_type']}, {row['evidence_grade']})")


PASS spectral_cube_inventory_14_entries
PASS spectral_member_spws_complete
PASS two_members_seven_targets_each
PASS x22_seven_of_seven_metadata_coverage
PASS x24_seven_of_seven_metadata_coverage
PASS historical_spw_all_reassigned
PASS x22_36394589_uses_spw25
PASS x24_36394589_uses_spw25
PASS doublet_second_axis_only
PASS all_matched_windows_covered
PASS species_not_independently_certified
COVERAGE {'X22': 7, 'X24': 7}
X22 351.76864 GHz: frozen SPW27 -> header SPW35 (HEADER_REST_MATCH, ARCHIVED_H0_FITS_HEADER_AUDIT)
X22 362.63030 GHz: frozen SPW29 -> header SPW27 (HEADER_REST_MATCH, ARCHIVED_H0_FITS_HEADER_AUDIT)
X22 362.73605 GHz: frozen SPW31 -> header SPW29 (HEADER_REST_MATCH, ARCHIVED_H0_FITS_HEADER_AUDIT)
X22 363.94589 GHz: frozen SPW33 -> header SPW25 (HEADER_REST_MATCH, USER_COLAB_REPORTED_FITS_HEADER_AND_SHA256__DRIVE_FILE_INVENTORY_CONFIRMED__NOT_INDEPENDENTLY_HASHED_THIS_RUN)
X22 364.10325 GHz: frozen SPW35 -> header SPW33 (HEADER_REST_MATCH, ARCHIVED_H0_FITS_HEADER_AUDIT)
X22

## Amendment gate and explicit blockers

All 14 target-frequency/member pairs now have a candidate whose frequency axis includes both **pre-frozen** velocity windows. For X22 SPW25 this relies on an archived Colab report and present Drive listing; it is not an independently recomputed SHA in the present environment. Original six-SPW assignments failed against headers and `spw25_primary_test` was false, so accepting the mapping would change the **analysis plan**. The amendment is therefore submitted as a separate versioned proposal requiring the researcher's explicit approval. A `PASS` in an arithmetical gate cannot authorize unblinding.

Additional provenance: only the official X22 README was retrieved in the inspected README folder; X24-specific README/QA2 not independently archived there. The preregistration file inspected here does not itself specify an explicit quantitative absorption decision threshold or negative controls; other unseen documents may exist. Freeze these elements or document an existing preregistration *before* viewing line amplitudes.

In [6]:
original_primary_misassigned=all(not row['original_spw_still_matches'] for row in row_out)
report_sha_independently_rechecked=False
missing_x24_member_readme = not archive_recovery['X24_readme_present']
thresholds_frozen_in_inspected_contract = bool(FROZEN_CONTRACT.get('line_test_thresholds'))
controls_frozen_in_inspected_contract = bool(FROZEN_CONTRACT.get('negative_controls'))

amendment={
 'id':'PATCH10H0R2_SPW_METADATA_MAPPING_PROPOSAL_v1_NOT_ADOPTED',
 'status':'RECOMMENDED_PENDING_EXPLICIT_AUTHOR_APPROVAL',
 'original_contract_sha256':EXPECTED_SOURCE_SHA['contract'],
 'original_spw_primary':[27,29,31,33,35,37],
 'proposed_exact_frequency_mapping':[{'target_ghz':r['target_GHz'],'original_spw':r['original_frozen_spw'],'proposed_spw':r['candidate_spw'],'match_type':r['match_type']} for r in row_out if r['member']=='X24'],
 'note_36428888':'FREQ axis inclusion in SPW31 is not header RESTFRQ match or independent transition identification',
 'spw25_added_as_primary':False,
 'spw25_for_36394589_recommended_conditionally':True,
 'line_thresholds_not_changed':True,
 'velocity_windows_unmodified':FROZEN_CONTRACT['velocity_windows_kms'],
 'original_SPW25_primary_test_False_preserved':True,
 'requires_explicit_author_approval_before_adoption':True,
 'author_approval_obtained_in_this_notebook':False,
 'line_signal_unblinded':False,
 'GVH_USED':False,
}

blocking_items=[
 'Author approval of a versioned SPW mapping amendment including secondary SPW25 use',
 'Independent spectroscopy/transition assignment and precise reference convention (esp. 364.28888 GHz axis-only component)',
 'Independent original X22 SPW25 byte-level checksum comparison or recorded accepted provenance standard',
 'Document X24 member README/QA2 provenance, or explicitly justify its non-requirement',
 'Locate and freeze quantitative detection thresholds and negative controls if absent from other original preregistration records',
]
gate('amendment_not_adopted',amendment['status']=='RECOMMENDED_PENDING_EXPLICIT_AUTHOR_APPROVAL')
gate('historic_spw25_exclusion_preserved',amendment['original_SPW25_primary_test_False_preserved'])
gate('metadata_compared_without_signal',not LINE_AMPLITUDES_INSPECTED)
gate('no_line_test_authorized',not LINE_TEST_AUTHORIZED)
gate('no_GVH_interpretation',not GVH_USED)
gate('no_retroactive_contract_mutation',source_audit['contract']['sha256']==EXPECTED_SOURCE_SHA['contract'])

final_status='PASS_METADATA_RECONCILIATION__AMENDMENT_RECOMMENDED_NOT_ADOPTED__ABSTAIN_LINE_TEST'
results={
 'patch':'GCOL-OBS-PATCH.10H.0R.2',
 'status':final_status,
 'date':'2026-11-20',
 'source_scope':'Direct local small FITS headers + archived primary header CSV + Drive listing + X22 SPW25 user-Colab report; no spectral amplitudes',
 'drive_inventory':DRIVE_INVENTORY_SNAPSHOT,
 'source_integrity':source_audit,
 'recovery':archive_recovery,
 'spatial_fits_direct':{k:v for k,v in small_header.items() if k.startswith('cont')},
 'spw25_x24_direct_header':small_header.get('spw25x24',{}),
 'x22_spw25_colab_reported':{'sha256_reported':x22_sha,'header':x22_25,'raw_bytes_independently_rechecked_here':report_sha_independently_rechecked},
 'mapping':row_out,
 'coverage_per_member':coverage,
 'all_original_primary_spw_assignments_mismatched':original_primary_misassigned,
 'amendment':amendment,
 'blocking_items':blocking_items,
 'frozen_contract_unchanged':True,
 'line_signals_inspected':False,
 'line_test_authorized':False,
 'GVH_USED':False,
 'IRAS4_FOREGROUND_LAYER_CERTIFIED':False,
 'FRONT_BACK_LAYER_ORDER_DERIVED':False,
 'checks':checks,
 'checks_total':len(checks),
 'checks_passed':sum(k['passed'] for k in checks.values()),
}
assert results['checks_total']==results['checks_passed']
out=OUTPUT/(STEM+'_results.json')
out.write_text(json.dumps(results,indent=2,ensure_ascii=False),encoding='utf-8')
csv_path=OUTPUT/(STEM+'_mapping_evidence.csv')
with csv_path.open('w',newline='',encoding='utf-8') as fh:
    w=csv.DictWriter(fh,fieldnames=list(row_out[0]));w.writeheader();w.writerows(row_out)
(OUTPUT/(STEM+'_amendment_PROPOSED_NOT_ADOPTED.json')).write_text(json.dumps(amendment,indent=2,ensure_ascii=False),encoding='utf-8')
print('DECISION',final_status)
print('VALIDATED CHECKS',results['checks_passed'],'/',results['checks_total'])
print('OUTPUT',out)


PASS amendment_not_adopted
PASS historic_spw25_exclusion_preserved
PASS metadata_compared_without_signal
PASS no_line_test_authorized
PASS no_GVH_interpretation
PASS no_retroactive_contract_mutation
DECISION PASS_METADATA_RECONCILIATION__AMENDMENT_RECOMMENDED_NOT_ADOPTED__ABSTAIN_LINE_TEST
VALIDATED CHECKS 64 / 64
OUTPUT /mnt/data/GCOL_OBS_PATCH10H0R2_Full_X22_X24_Metadata_And_Amendment_Decision_Gate_FAST_results.json


## Level 4 — scientific decision

**Metadata alignment:** `PASS` (7/7 nominal target frequencies per member with explicit provenance distinctions). **Physical/chemical identification:** not demonstrated by these headers alone. **Amendment:** proposed only; requires author sign-off and supporting material. **Line test:** `ABSTAIN`; no optical depth, layer order or GVH conclusion. Previous historical FAIL/ABSTAIN decisions are retained in the provenance chain.

*Next authorized task:* obtain scientific author decision on the amendment after independent rest-frequency/source mapping review, document X24 README/QA2 and comparison of the X22 SPW25 whole-file hash with an independent trusted source; audit quantitative thresholds/negative controls. Only then draft a new, prospectively frozen line-test protocol. Do **not** use the present notebook as implicit authorization to unblind.